<a href="https://colab.research.google.com/github/tk-learning-artifacts/ai-accelerator-2026-notebooks/blob/main/student/d2-first-network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 2 · Notebook 4 — Your first network in PyTorch

This morning you wrote gradient descent for two numbers. This afternoon you
use **PyTorch**, the library real models are built in, to train a network
with a few thousand numbers to read handwritten digits. The loop is the one you
already know: predict, measure the loss, find the slopes, step downhill.
PyTorch does the slopes for you.

Same rules: 🔮 predict, ✏️ replace the `...`, ✅ check, 💥 break it. Swap
driver and navigator at every `##`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.datasets import load_digits


def is_done(*values):
    """True once none of the values is still `...`."""
    return all(v is not ... for v in values)


def check(ok, hint):
    if not ok:
        raise AssertionError("Not yet. " + hint)
    print("✅ Correct!")


torch.manual_seed(0)
print("PyTorch", torch.__version__, "ready.")

## 1. Tensors: arrays that remember how they were made

The data: 1,797 handwritten digits, each a tiny 8×8 greyscale image, with the
right answer for each.

In [ ]:
digits = load_digits()
images, labels = digits.images, digits.target

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow(images[i], cmap="gray_r")
    ax.set_title(labels[i])
    ax.axis("off")
plt.show()

🔮 **Predict.** What is `images.shape`? What is the largest pixel value? (These
are not 0–255 images.)

In [ ]:
my_prediction = ""  # @param {type:"string"}

In [ ]:
print("images:", images.shape, images.dtype, " largest value:", images.max())
print("labels:", labels.shape, " first ten:", labels[:10])

PyTorch works on **tensors**. A tensor is a NumPy array that can also live on
a GPU and remember the calculations done to it, which is how PyTorch finds
slopes for you.

✏️ **Exercise 1a.** Make `X`, a tensor of the images with `dtype=torch.float32`
and every value divided by 16 so it runs from 0 to 1. Make `y`, a tensor of the
labels (`torch.tensor(labels)` is enough).

In [ ]:
X = ...
y = ...

In [ ]:
check(is_done(X, y), "Replace both ... .")
check(isinstance(X, torch.Tensor) and X.dtype == torch.float32, "X should be a float32 tensor.")
check(tuple(X.shape) == (1797, 8, 8), f"X should keep the shape (1797, 8, 8), but it is {tuple(X.shape)}.")
check(np.isclose(X.max().item(), 1.0), "Divide by 16 so the largest value is 1.")
check(isinstance(y, torch.Tensor) and y.shape[0] == 1797, "y should be a tensor with one label per image.")

Everyone continues from the same place: this cell makes `X` and `y` another
way (`torch.from_numpy` shares memory with the NumPy array instead of
copying it).

In [ ]:
X = torch.from_numpy(images).float() / 16
y = torch.from_numpy(labels)

## 2. Ten-year series, prelims, O-levels

Before training we split the data three ways, like this morning:

- **train** (1,200 images) is the ten-year series: the model practises on it
  as often as it likes;
- **validation** (300) is the prelims: we check progress on it and change our
  settings based on the result;
- **test** (the last 297) is the O-levels: used **once**, at the very end.

In [ ]:
order = torch.randperm(1797, generator=torch.Generator().manual_seed(0))
train_idx, val_idx, test_idx = order[:1200], order[1200:1500], order[1500:]
print(len(train_idx), len(val_idx), len(test_idx))

🔮 **Predict.** Why shuffle with `randperm` instead of taking the first 1,200
images as they come? And why must no image appear in two of the three groups?

In [ ]:
my_prediction = ""  # @param {type:"string"}

✏️ **Exercise 2a.** Prove the split is fair. Write `count_overlap(a, b)`: how
many image numbers are in both index tensors `a` and `b`. (Turn each into a
Python `set` with `set(a.tolist())`; then `&` gives the shared ones.)

In [ ]:
def count_overlap(a, b):
    return ...

In [ ]:
leaky_test = order[1100:1400]   # a badly made split that reuses 100 training images
check(is_done(count_overlap(train_idx, test_idx)), "count_overlap still returns ... .")
check(count_overlap(train_idx, leaky_test) == 100, "This leaky split shares 100 images with train; your function should find all 100.")
check(count_overlap(train_idx, test_idx) == 0, "Our real split should share nothing.")
print("Overlap of train and test:", count_overlap(train_idx, test_idx))

## 3. The training step, in order

Here is the network: flatten the 8×8 image into 64 numbers, a layer of
neurons with ReLU (this morning's neuron, many times over), and 10 outputs,
one score per digit.

In [ ]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 10),
)
print(model)
print("numbers to learn:", sum(p.numel() for p in model.parameters()))

One training step is five lines. Here they are, **shuffled**:

- **A** `optimizer.step()`
- **B** `loss = loss_fn(scores, y_batch)`
- **C** `optimizer.zero_grad()`
- **D** `scores = model(x_batch)`
- **E** `loss.backward()`

✏️ **Exercise 3a (Parsons problem).** Put them in a working order. Talk it
through with your partner first: what does each line need to exist before it
can run? Then write the letters as a string, like `"ABCDE"`.

In [ ]:
step_order = "..."

In [ ]:
s = step_order.upper().replace(" ", "")
check(sorted(s) == list("ABCDE"), "Use each of the five letters once.")
pos = {letter: s.index(letter) for letter in s}
check(pos["D"] < pos["B"], "The loss compares the model's scores with the answers. Where do the scores come from?")
check(pos["B"] < pos["E"], "backward() finds the slopes of the loss. Does the loss exist yet?")
check(pos["E"] < pos["A"], "step() moves downhill using the slopes. Have they been found yet?")
check(pos["C"] < pos["E"], "zero_grad() clears the old slopes. If it runs after backward(), it wipes the new ones.")

Here is the whole training loop, with your five lines inside it.

In [ ]:
def train(model, epochs, lr, idx=train_idx, batch=64, do_step=True):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    history = {"train": [], "val": []}
    for epoch in range(epochs):
        shuffled = idx[torch.randperm(len(idx))]
        for start in range(0, len(shuffled), batch):
            b = shuffled[start:start + batch]
            x_batch, y_batch = X[b], y[b]
            optimizer.zero_grad()
            scores = model(x_batch)
            loss = loss_fn(scores, y_batch)
            loss.backward()
            if do_step:
                optimizer.step()
        with torch.no_grad():
            history["train"].append(loss_fn(model(X[idx]), y[idx]).item())
            history["val"].append(loss_fn(model(X[val_idx]), y[val_idx]).item())
    return history

💥 **Break it.** 🔮 Predict first: if `optimizer.step()` never runs, what
happens to the loss? Then change `do_step=True` to `do_step=False` below (the
loop above skips `optimizer.step()` when it is `False`) and run it.

In [ ]:
torch.manual_seed(0)
test_model = nn.Sequential(nn.Flatten(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
h = train(test_model, epochs=5, lr=0.1, do_step=True)
print("training loss after each epoch:", np.round(h["train"], 3))

## 4. Train it: beat 90%

The network outputs 10 scores; its answer is the digit with the highest
score. **Accuracy** is the fraction of answers that are right.

✏️ **Exercise 4a.** Write `accuracy(scores, answers)`. `scores.argmax(dim=1)`
gives the highest-scoring digit for each image; compare with `answers`, and
take the mean of the result as a float.

In [ ]:
def accuracy(scores, answers):
    return ...

In [ ]:
fake_scores = torch.tensor([[0.1, 0.9], [0.8, 0.2], [0.3, 0.7], [0.6, 0.4]])
check(is_done(accuracy(fake_scores, torch.tensor([1, 0, 0, 0]))), "accuracy still returns ... .")
check(np.isclose(accuracy(fake_scores, torch.tensor([1, 0, 0, 0])), 0.75), "Three of these four answers are right, so 0.75.")
check(isinstance(accuracy(fake_scores, torch.tensor([1, 0, 1, 0])), float), "Return a plain number: finish with .item().")

✏️ **Exercise 4b.** Choose settings that get **validation** accuracy above
90%. Change one thing at a time and write down what each change did.

In [ ]:
hidden = 8  # @param {type:"integer"}
lr = 0.01  # @param {type:"number"}
epochs = 5  # @param {type:"integer"}
torch.manual_seed(0)
model = nn.Sequential(nn.Flatten(), nn.Linear(64, hidden), nn.ReLU(), nn.Linear(hidden, 10))
history = train(model, epochs=epochs, lr=lr)
with torch.no_grad():
    val_acc = accuracy(model(X[val_idx]), y[val_idx])
print(f"hidden={hidden}, lr={lr}, epochs={epochs}: validation accuracy {val_acc:.1%}")
plt.plot(history["train"], label="train")
plt.plot(history["val"], label="validation")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()

In [ ]:
check(val_acc > 0.9, f"Validation accuracy is {val_acc:.1%}. Try more hidden neurons, a bigger lr, or more epochs.")

Now, **once**, the O-level: the test set.

In [ ]:
with torch.no_grad():
    test_acc = accuracy(model(X[test_idx]), y[test_idx])
print(f"Test accuracy: {test_acc:.1%}  (validation was {val_acc:.1%})")

🔮 **Think.** Is your test accuracy higher or lower than validation? If you had
tried 50 settings and picked the best validation score, which of the two
numbers would you trust more, and why?

## 5. Overfit on purpose

A model can learn the practice papers by heart instead of learning the
subject. Let's make that happen: a big network, only 40 training images, and
far too many epochs. Run it as given first; then each pair changes one of the
three settings and shows the class what happened to the gap between the
curves.

🔮 **Predict.** Sketch on paper what you expect the train and validation loss
curves to do.

In [ ]:
my_prediction = ""  # @param {type:"string"}

In [ ]:
n_images = 40  # @param {type:"integer"}
hidden_size = 256  # @param {type:"integer"}
n_epochs = 300  # @param {type:"integer"}
torch.manual_seed(0)
big = nn.Sequential(nn.Flatten(), nn.Linear(64, hidden_size), nn.ReLU(), nn.Linear(hidden_size, 10))
h = train(big, epochs=n_epochs, lr=0.5, idx=train_idx[:n_images])
with torch.no_grad():
    train_acc = (big(X[train_idx[:n_images]]).argmax(dim=1) == y[train_idx[:n_images]]).float().mean().item()
    overfit_val_acc = (big(X[val_idx]).argmax(dim=1) == y[val_idx]).float().mean().item()
print(f"train accuracy {train_acc:.0%}, validation accuracy {overfit_val_acc:.0%}")
plt.plot(h["train"], label=f"train ({n_images} images)")
plt.plot(h["val"], label="validation")
plt.yscale("log")   # a log scale makes small changes near the bottom visible
plt.xlabel("epoch")
plt.ylabel("loss (log scale)")
plt.legend()
plt.show()

🔮 **Name it.** Choose what this picture shows, and what you would do about
it.

In [ ]:
diagnosis = "?"  # @param ["?", "underfitting", "a good fit", "overfitting"]
fix = ""  # @param {type:"string"}

## Extension (if you finish early)

Open the Day 2 extension track in Sidecar, or:

- Find the smallest `hidden` that still beats 90%. How many numbers does that
  model have?
- Train with 40, 100, 300 and 1,200 training images and plot validation
  accuracy against the number of images.
- Show the test images the model gets wrong (about a dozen), with its answer and the right
  one. Would you have got them right?

In [ ]:
# Your extension work here.